# Prétraitement du dataset Diabète – Sprint 1
## Contexte du projet (énoncé officiel)
Le projet vise la prédiction du diabète à partir des données BRFSS 2015 (CDC/Kaggle), avec un enjeu de santé publique fort : améliorer le repérage précoce des patients à risque.

Données de départ (énoncé) :

 - 253 680 répondants,
 - 22 variables,
 - objectif final de modélisation : classification binaire, évaluée principalement en ROC AUC.

## Objectif de ce notebook
Ce notebook couvre la partie Sprint 1 – Prétraitement pour produire un jeu de données propre, traçable et réutilisable par les sprints de modélisation.

Pipeline implémenté ici :

 1. Lecture et inspection du dataset source,
 2. Séparation cible / variables explicatives,
 3. Découpage train / validation / test,
 4. Normalisation des variables,
 5. Sauvegarde des artefacts.


## Livrables produits
 - data/processed/train.parquet
 - data/processed/val.parquet
 - data/processed/test.parquet
 - artifacts/scaler.joblib
Ces sorties permettent de lancer l’entraînement de manière reproductible et cohérente avec les objectifs de l’énoncé.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

TARGET_COL = "Diabetes_binary"

## Lecture du dataset
On charge le fichier CSV et on affiche un aperçu ainsi que les informations générales.

In [ ]:
def read_data(file_path="data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"):
    try:
        df = pd.read_csv(file_path)
        print("Aperçu des 5 premières lignes :")
        print(df.head())
        print("\nInformations générales :")
        print(df.info())
        print("\nValeurs manquantes par colonne :")
        print(df.isnull().sum())
        df.to_parquet("test_index_false.parquet", index=False)
        return df
    except FileNotFoundError:
        print(f"Erreur : le fichier {file_path} n'a pas été trouvé.")
        return None

df = read_data()

## Prétraitement initial
- Regroupement des classes : 0 = no diabète ou prediabète, 1 = diabète
- Séparation des features (X) et de la cible (y)

## Typage des variables 
Dans cette partie il est question pour nous de faire une sépartion des variables selon leurs types pour faciliter l'analyse exploratoire. 
On va donc faire deux dataset qui seront utiles uniquement pour l'analyse exploratoire dont un pour les variables numérique et l'autre pour les variables binaires (dans notre cas).

In [ ]:
# Typage des variables 
# Colonnes binaires (0/1)
binary_cols = [
    col for col in df.columns
    if df[col].nunique() == 2 and col != "Diabetes_binary"
]

print("Colonnes binaires :", binary_cols)


# Colonnes ordinales (entiers avec peu de modalités)
ordinal_cols = [
    col for col in df.columns
    if col not in binary_cols + ["Diabetes_binary"]
    and df[col].nunique() <= 15
]

print("Colonnes ordinales :", ordinal_cols)


# Colonnes numériques continues
numeric_cols = [
    col for col in df.columns
    if col not in binary_cols + ordinal_cols + ["Diabetes_binary"]
]

print("Colonnes numériques :", numeric_cols)

# Conversion en float
for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

    # Conversion en Binaire
for col in binary_cols:
        df[col] = df[col].apply(lambda x: True if x not in [0, '0', False, 'False', None, np.nan] else False)
        df[col] = df[col].astype(bool)

for col in ordinal_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

print("\nTypes après conversion :")
print(df.dtypes)

## Nettoyage des doublons et gestion des valeurs manquantes
- Suppression des doublons
- Gestion des valeurs manquantes (imputation ou suppression selon le cas)

In [ ]:
print(f"Nombre de lignes avant suppression des doublons : {len(df)}")

# Affichage des valeurs manquantes et des doublons 
print("=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Aucune valeur manquante ")

print("\n=== Doublons ===")
n_duplicates = df.duplicated().sum()
print(f"Nombre de lignes dupliquées : {n_duplicates:,}")

# Suppression des doublons
print("\n=== Suppression des doublons ===")

if n_duplicates > 0:
    df_clean = df.drop_duplicates()
    print(f"Nombre de lignes après suppression des doublons : {len(df)}")
    df = df_clean
else:
    print("Aucun doublon à supprimer.")

In [ ]:
def separate_data(df):
    X = df.drop(TARGET_COL, axis=1)
    y = df[TARGET_COL]
    return X, y

if df is not None:
    X, y = separate_data(df)
    print("Distribution des classes :")
    print(y.value_counts())

## Séparation Train / Validation / Test
- 70% train
- 15% validation
- 15% test
- Stratification pour garder la proportion des classes

In [ ]:
if df is not None:
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    print("Taille train :", X_train.shape)
    print("Taille validation :", X_val.shape)
    print("Taille test :", X_test.shape)
    print(f"nb de ligne : {len(df)}")


## EDA (Exploratory Data Analysis)
### Ce que traite cette étape 

Dans cette étape on va faire une visualisation globale de nos différente variable, on va essayer de voir la distribution des differentes et premièrement celle de la varible cible 

  **Pourquoi et en quoi est ce utile?**  

#### Distribution de la variable cible 

On visualise la répartition des classes (`0 = Non-Diabétique`, `1 = Diabétique`) avec **deux graphiques côte à côte** :
- **Bar chart** : nombre exact de patients par classe avec pourcentages
- **Pie chart** : proportion relative des deux classes



In [ ]:
# création des datasets numérique et binaire
df_numeric = df[numeric_cols]
df_ordinal = df[ordinal_cols]
df_binary = df[binary_cols ]

In [ ]:
counts = df["Diabetes_binary"].value_counts().sort_index()
labels = ["Non diabétique (0)", "Diabétique (1)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribution de la variable cible 'Diabetes_binary'", fontsize=16, fontweight='bold',color='darkblue', y=1.02)

# Bar Plot
bars = axes[0].bar(labels, counts.values, color=['skyblue', 'salmon'], edgecolor='black', linewidth=1.5, width=0.6)
axes[0].set_title("Nombre de patients par classe", fontsize=14, fontweight='bold', color='darkblue')
axes[0].set_ylabel("Nombre de patients", fontsize=12)

for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + len(df)*0.005, f'{val: ,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom',color='black',fontweight='bold', fontsize=12)
    
axes[0].set_ylim(0, counts.max() * 1.2)
axes[0].grid(axis='y', alpha=0.6)


# Pie Chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=labels, autopct='%1.1f%%', 
    colors=['skyblue', 'salmon'], startangle=90,
    textprops={'color':"black", 'fontweight':'bold', 'fontsize':12}, 
    wedgeprops={'edgecolor': 'black', 'linewidth': 2.5},
    pctdistance=0.8
    )

for at  in autotexts:
    at.set_fontsize(14)
    at.set_color('black')
    at.set_fontweight('bold')
axes[1].set_title("Proportion relative des classes", fontsize=14, fontweight='bold', color='darkblue')

plt.tight_layout()
plt.show()


ratio = counts.max() / counts.min()
print(f' classe 0 : {counts[0]:,} patients ({counts[0]/len(df)*100:.1f}%)')
print(f' classe 1 : {counts[1]:,} patients ({counts[1]/len(df)*100:.1f}%)')
print(f' Ratio de déséquilibre : {ratio:.2f}')
if ratio > 1.5:
    print('Déséquilibre significatif détecté. Envisagez des techniques de rééchantillonnage ou de pondération pour améliorer les performances du modèle.')
else:
    print('classes relativement équilibrées. Les techniques de rééchantillonnage peuvent ne pas être nécessaires, mais surveillez les performances du modèle sur la classe minoritaire.')

## Analyse exploratoire des variables numériques


In [ ]:
n = len(df_numeric.columns)
ncols = n
nrows = 3

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 12))
fig.patch.set_facecolor('white')

fig.suptitle(
    'Analyse des Variables Numériques par Classe',
    fontsize=16,
    fontweight='bold',
    color='black',
    y=1.02
)

# -------------------------
# LIGNE 1 — HISTOGRAMMES
# -------------------------
for i, col in enumerate(df_numeric.columns):
    ax = axes[0, i]

    for cls, color, lbl in zip(
        [0, 1],
        ['skyblue', 'salmon'],
        ['Non-Diabétique', 'Diabétique']
    ):
        subset = df[df['Diabetes_binary'] == cls][col].dropna()

        ax.hist(
            subset,
            bins=20,
            alpha=0.6,
            color=color,
            label=lbl,
            edgecolor='black'
        )

        ax.axvline(
            subset.median(),
            color=color,
            linestyle='--',
            linewidth=1.5
        )

    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col, color='black', fontsize=9)
    ax.set_ylabel("Fréquence")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)


# -------------------------
# LIGNE 2 — BOXPLOTS
# -------------------------
for i, col in enumerate(df_numeric.columns):
    ax = axes[1, i]

    data = [
        df[df['Diabetes_binary'] == 0][col].dropna(),
        df[df['Diabetes_binary'] == 1][col].dropna()
    ]

    bp = ax.boxplot(
        data,
        patch_artist=True,
        medianprops={'color': 'black', 'linewidth': 2.5},
        whiskerprops={'color': 'black', 'linewidth': 1.2},
        capprops={'color': 'black', 'linewidth': 1.5},
        flierprops={'marker': 'o', 'markerfacecolor': '#FF6584',
                    'markersize': 2.5, 'alpha': 0.4, 'markeredgewidth': 0}
    )

    for patch, color in zip(bp['boxes'], ['skyblue', 'salmon']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Non-Diabétique', 'Diabétique'])
    ax.set_ylabel("Valeur")
    ax.grid(alpha=0.3)


# -------------------------
# LIGNE 3 — VIOLIN
# -------------------------
for i, col in enumerate(df_numeric.columns):
    ax = axes[2, i]

    data = [
        df[df['Diabetes_binary'] == 0][col].dropna(),
        df[df['Diabetes_binary'] == 1][col].dropna()
    ]

    vp = ax.violinplot(
        data,
        showmeans=True,
        showmedians=True
    )

    for body, color in zip(vp['bodies'], ['skyblue', 'salmon']):
        body.set_facecolor(color)
        body.set_alpha(0.7)
        body.set_edgecolor('black')

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Non-Diabétique', 'Diabétique'])
    ax.set_ylabel("Valeur")
    ax.grid(alpha=0.3)


plt.tight_layout()
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.show()

## Analyse exploratoire des variables binaires 

In [ ]:
n      = len(df_binary.columns)
ncols  = 4
nrows  = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
fig.patch.set_facecolor('white')
fig.suptitle('Variables Binaires par Classe',
             fontsize=16, fontweight='bold', color='black', y=1.01)

axes = axes.flatten()

for i, col in enumerate(df_binary.columns):
    ax = axes[i]
    
    prop = pd.crosstab(df[col], df['Diabetes_binary'])
    prop.plot(kind='bar', ax=ax, alpha=0.8, color=['skyblue', 'salmon'], edgecolor='black', linewidth=1.5, width=0.6)
    
    ax.set_title(col, color='black', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Proportion')
    ax.legend(labels=['Non-Diabétique', 'Diabétique'])
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Analyse exploratoire des variables ordinales

In [ ]:
n      = len(df_ordinal.columns)
ncols  = 4
nrows  = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 5))
fig.patch.set_facecolor('white')
fig.suptitle('Variables Ordinales par Classe',
             fontsize=16, fontweight='bold', color='black', y=1.01)

axes = axes.flatten()

for i, col in enumerate(df_ordinal.columns):
    ax = axes[i]
    
    prop = pd.crosstab(df[col], df['Diabetes_binary'], normalize='index')
    prop.plot(kind='bar', ax=ax, alpha=0.8)
    
    ax.set_title(col, color='black', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Proportion')
    ax.legend(labels=['Non-Diabétique', 'Diabétique'])
ax.grid(True, alpha=0.3)





plt.tight_layout()
plt.show()

In [ ]:
all_cols = numeric_cols + ordinal_cols + binary_cols + [TARGET_COL]
train_analysis_df = X_train.copy()
train_analysis_df[TARGET_COL] = y_train.values
corr_matrix = train_analysis_df[all_cols].corr()

fig, ax = plt.subplots(figsize=(18,14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    linecolor='white', ax=ax,
    annot_kws={'size': 9, 'weight': 'bold'},
    cbar_kws={'shrink': 0.8, 'label': 'Corrélation de Pearson'},
    vmin=-1, vmax=1
)
ax.set_title('Matrice de Corrélation (Train uniquement)', fontsize=18, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## Choix des variables 

> Dans la partie qui suit on s'intérresse à l'importance des variables pour faire un choix intellingent des variables qui seront utile pour l'entrainement de notre modèle tout en évitant les risques de surapprentissage 


In [ ]:
target_corr = corr_matrix[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)

plt.figure(figsize=(14, 7))

sns.barplot(
    x=target_corr.values, 
    y=target_corr.index, 
    hue=target_corr.index,
    palette='coolwarm',
    legend=False
)

plt.title(f'Corrélation absolue avec la cible (Train) : {TARGET_COL}', fontsize=15, pad=20)
plt.xlabel('Score de corrélation (Valeur absolue)', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
CORR_TARGET_THRESHOLD = 0.05
CORR_INTER_THRESHOLD = 0.60


weak_features = target_corr[target_corr < CORR_TARGET_THRESHOLD].index.tolist()


to_drop_redundant = set()
feature_corr = corr_matrix.drop(index=TARGET_COL, columns=TARGET_COL)
for i in range(len(feature_corr.columns)):
    for j in range(i):
        if abs(feature_corr.iloc[i, j]) > CORR_INTER_THRESHOLD:
            colname_i = feature_corr.columns[i]
            colname_j = feature_corr.columns[j]

            if target_corr[colname_i] > target_corr[colname_j]:
                to_drop_redundant.add(colname_j)
            else:
                to_drop_redundant.add(colname_i)


all_to_drop = set(weak_features) | to_drop_redundant
final_features = [c for c in target_corr.index if c not in all_to_drop]


print("## RAPPORT DE SÉLECTION DES VARIABLES (Train) ##\n")
print(f" À SUPPRIMER (Faible impact cible < {CORR_TARGET_THRESHOLD}):")
print(f"   -> {weak_features if weak_features else 'Aucune'}")
print("\n À SUPPRIMER (Redondance inter-features):")
print(f"   -> {sorted(to_drop_redundant) if to_drop_redundant else 'Aucune'}")
print("\n À GARDER (Variables finales sélectionnées):")
print(f"   -> {final_features}")


La sélection est réalisée sur le jeu d'entraînement uniquement, à partir de la corrélation absolue avec la cible et d'un filtrage de redondance entre variables.

>NB: les seuils peuvent être ajustés selon les besoins métier et les résultats de validation.

In [ ]:
PALETTE_MAIN = ['#74b9ff', '#ff7675'] 


top_features = target_corr.nlargest(5).index.tolist()
print(f'Top 5 features sélectionnées : {top_features}')


pairplot_df = train_analysis_df[top_features + [TARGET_COL]].copy()
pairplot_df[TARGET_COL] = pairplot_df[TARGET_COL].map(
    {0: 'Non-Diabétique', 1: 'Diabétique'}
)


g = sns.pairplot(
    pairplot_df, hue=TARGET_COL,
    palette={'Non-Diabétique': PALETTE_MAIN[0], 'Diabétique': PALETTE_MAIN[1]},
    diag_kind='kde',
    plot_kws={'alpha': 0.35, 's': 12, 'edgecolor': 'none'},
    diag_kws={'fill': True, 'alpha': 0.4},
    corner=True,
)


g.figure.patch.set_facecolor('White')
g.figure.suptitle('Pairplot — Top 5 Features vs Classe Cible (Train)',
                  y=1.01, color='black', fontsize=14, fontweight='bold')

for ax in g.axes.flatten():
    if ax is not None:
        ax.set_facecolor('white')
        ax.tick_params(colors='black', labelsize=7)
        ax.xaxis.label.set_color('black')
        ax.yaxis.label.set_color('black')


plt.savefig('09_pairplot.png', dpi=130, bbox_inches='tight', facecolor='white')
plt.show()


>Ici on a un ensemble de graphe qui montre les varibles qui ont le plus de colinéarité avec notre cible  

## Normalisation des features
- Standardisation : moyenne=0, écart type=1
- Fit sur le train, transform sur validation et test

In [ ]:
# Application de la sélection finale avant normalisation
X_train = X_train[final_features]
X_val = X_val[final_features]
X_test = X_test[final_features]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Normalisation effectuée. Exemple :")
print(X_train_scaled[:5])

## Sauvegarde des datasets prétraités

Les datasets sont sauvegardés dans :

- `data/raw` : données brutes
- `data/processed` : données prétraitées

On sauvegarde :
- train (normalisé)
- validation (normalisé)
- test (normalisé)
- un dataset consolidé normalisé avec features sélectionnées + cible + split
- le scaler utilisé pour la normalisation

Cela permet de reproduire exactement le pipeline de preprocessing.

In [ ]:
import os
import joblib

# Création des dossiers projet
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

# Reconstruction des DataFrames à partir des données normalisées
df_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
df_train_scaled[TARGET_COL] = y_train.reset_index(drop=True)

df_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
df_val_scaled[TARGET_COL] = y_val.reset_index(drop=True)

df_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
df_test_scaled[TARGET_COL] = y_test.reset_index(drop=True)

# Sauvegarde des datasets normalisés par split
df_train_scaled.to_parquet("data/processed/train.parquet", index=False)
df_val_scaled.to_parquet("data/processed/val.parquet", index=False)
df_test_scaled.to_parquet("data/processed/test.parquet", index=False)

# Dataset consolidé normalisé (features sélectionnées + cible + split)
train_scaled_with_split = df_train_scaled.assign(split="train")
val_scaled_with_split = df_val_scaled.assign(split="val")
test_scaled_with_split = df_test_scaled.assign(split="test")

df_selected_consolidated_scaled = pd.concat(
    [train_scaled_with_split, val_scaled_with_split, test_scaled_with_split],
    axis=0,
    ignore_index=True,
)
df_selected_consolidated_scaled.to_parquet(
    "data/processed/selected_features_consolidated_scaled.parquet", index=False
)

# Sauvegarde des artefacts
joblib.dump(scaler, "artifacts/scaler.joblib")
joblib.dump(final_features, "artifacts/final_features.joblib")

print("Datasets normalisés sauvegardés dans data/processed/")
print("Dataset consolidé normalisé sauvegardé : data/processed/selected_features_consolidated_scaled.parquet")
print("Artefacts sauvegardés dans artifacts/")